Import lib

In [17]:
import argparse
import json
import os
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm import tqdm

Config

In [18]:
DATASET_ROOT = "/Users/tranminhhuy/Documents/dataset"
MODEL_NAME = "resnext"   # "resnext", "wideresnet", "efficientnet"
EPOCHS = 1 # look back train set
BATCH_SIZE = 8
IMG_SIZE = 224
LR = 0.01 # learning rate -> based on original paper
MOMENTUM = 0.9 # based on original paper
TRAIN_RATIO = 0.7 # percentage dataset used to train
SEED = 42 # for random
NUM_WORKERS = 0 # worker helps to load data
PRETRAINED = False # model does not use data learnt before

OUTPUT_DIR = Path("./outputs_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Seed & device

In [19]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()
print("Using device:", device)

Using device: mps


Transforms

In [20]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

Load dataset

In [22]:
base_dataset = datasets.ImageFolder(DATASET_ROOT)
train_dataset_full = datasets.ImageFolder(DATASET_ROOT, transform=train_transform)
test_dataset_full = datasets.ImageFolder(DATASET_ROOT, transform=test_transform)

print("Classes:", base_dataset.classes)
print("Num classes:", len(base_dataset.classes))
print("Total images:", len(base_dataset))

Classes: ['aerial', 'architecture', 'event', 'fashion', 'food', 'nature', 'sports', 'street', 'wedding', 'wildlife']
Num classes: 10
Total images: 100


Split dataset with 70% train and 30% test

In [23]:
from collections import defaultdict
import random

def stratified_split_indices(targets, train_ratio=0.7, seed=42):
    rng = random.Random(seed)
    class_to_indices = defaultdict(list)

    for idx, label in enumerate(targets):
        class_to_indices[label].append(idx)

    train_indices = []
    test_indices = []

    for label, indices in class_to_indices.items():
        indices = indices[:]
        rng.shuffle(indices)
        split_idx = int(len(indices) * train_ratio)

        train_indices.extend(indices[:split_idx])
        test_indices.extend(indices[split_idx:])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return train_indices, test_indices


train_indices, test_indices = stratified_split_indices(
    base_dataset.targets,
    train_ratio=TRAIN_RATIO,
    seed=SEED
)

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Train samples: 70
Test samples: 30


Dataloader prepare data so that model can read it in batches

In [24]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Train loader batches:", len(train_loader))
print("Test loader batches:", len(test_loader))

Train loader batches: 9
Test loader batches: 4
